# Vẽ hình từ kết quả TV2 đã lưu
Notebook độc lập, không huấn luyện. Thêm dataset `tv2-results` đã giải nén vào Kaggle Input.
Sửa `source_dir` nếu đường dẫn dataset khác; đầu ra `/kaggle/working/TV2_RESTORED/figures_report`.
Môi trường khuyến nghị Python 3.13; cài thư viện rồi Run All.


In [ ]:
%pip install -q numpy==2.1.3 pandas==2.3.3 matplotlib==3.10.0


In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

prediction_files = sorted(
    input_root.rglob("ds2_predictions.csv")
)

zip_files = sorted(
    path
    for path in input_root.rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".zip"
    and "tv2" in path.name.lower()
)

if prediction_files:
    for path in prediction_files:
        print("Tìm thấy kết quả:", path)
elif zip_files:
    for path in zip_files:
        print("Tìm thấy ZIP:", path)
else:
    print("Chưa tìm thấy kết quả TV2. Kiểm tra đã thêm dataset vào Input chưa.")

In [ ]:
from pathlib import Path
import shutil

source_dir = Path(
    "/kaggle/input/datasets/quanghuy10/tv2-results"
)

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")

required_files = [
    "results/ds2_predictions.csv",
    "results/ds2_metrics.csv",
    "results/conversion_metrics.csv",
    "results/model_sizes.csv",
    "logs/lr_001_baseline_seed42_fold1.csv",
    "logs/lr_001_cnn_rr_seed42_fold1.csv",
]

missing = [
    name for name in required_files
    if not (source_dir / name).is_file()
]

if missing:
    raise FileNotFoundError(
        f"Dataset thiếu các tệp: {missing}"
    )

shutil.copytree(
    source_dir,
    OUT_DIR,
    dirs_exist_ok=True,
)

print("Đã khôi phục kết quả tại:", OUT_DIR)
print("Số tệp CSV:", len(list(OUT_DIR.rglob("*.csv"))))
print("Số hình saliency:", len(list((OUT_DIR / "saliency").glob("*.png"))))
print("Sẵn sàng tạo biểu đồ, không cần train lại.")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

models = {
    "baseline": "CNN cơ sở",
    "cnn_rr": "CNN + RR",
}

for variant, model_name in models.items():
    history_path = (
        OUT_DIR / "logs"
        / f"lr_001_{variant}_seed42_fold1.csv"
    )

    history = pd.read_csv(history_path).sort_values("epoch")

    required = {"epoch", "train_loss", "val_loss", "f1_macro"}

    if not required.issubset(history.columns):
        raise ValueError(
            f"{history_path.name} thiếu cột: "
            f"{sorted(required - set(history.columns))}"
        )

    best_row = history.loc[history["f1_macro"].idxmax()]
    best_epoch = int(best_row["epoch"])
    best_f1 = float(best_row["f1_macro"])

    figure, axes = plt.subplots(
        1, 2,
        figsize=(12, 5),
        constrained_layout=True,
    )

    axes[0].plot(
        history["epoch"],
        history["train_loss"],
        marker="o",
        color="#2563EB",
        label="Train loss có trọng số",
    )

    axes[0].plot(
        history["epoch"],
        history["val_loss"],
        marker="s",
        color="#EA580C",
        label="Validation loss không trọng số",
    )

    axes[0].set_title("Diễn biến loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.25)

    axes[1].plot(
        history["epoch"],
        history["f1_macro"],
        marker="o",
        color="#15803D",
        label="F1 macro validation",
    )

    axes[1].axvline(
        best_epoch,
        color="#DC2626",
        linestyle="--",
        label=f"Checkpoint tốt nhất: epoch {best_epoch}",
    )

    axes[1].scatter(
        [best_epoch],
        [best_f1],
        color="#DC2626",
        s=65,
        zorder=5,
    )

    axes[1].set_title("Diễn biến F1 macro")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1 macro validation")
    axes[1].set_ylim(0, 1)
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.25)

    for axis in axes:
        axis.set_xticks(history["epoch"].astype(int))
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    figure.suptitle(
        f"{model_name} — DS1 — seed 42 — fold 1 — learning rate 0.001",
        fontsize=13,
        fontweight="bold",
    )

    filename = f"training_{variant}_seed42_fold1"

    figure.savefig(
        FIG_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIG_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    print(
        f"{model_name}: epoch tốt nhất = {best_epoch}, "
        f"F1 macro validation = {best_f1:.4f}"
    )

print("\nĐã lưu 2 hình PNG và 2 bản PDF tại:")
print(FIG_DIR)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

class_names = ["N", "S", "V", "F", "Q"]
model_names = {
    "baseline": "CNN cơ sở",
    "cnn_rr": "CNN + RR",
}
seeds = [42, 123, 2026]

saved_count = 0

for variant, model_name in model_names.items():
    for seed in seeds:
        matrix_path = (
            OUT_DIR / "results"
            / f"confusion_{variant}_seed{seed}.csv"
        )

        table = pd.read_csv(matrix_path)

        matrix = (
            table.set_index("true_class")
            .loc[class_names, class_names]
            .to_numpy(dtype=np.int64)
        )

        if matrix.shape != (5, 5) or np.any(matrix < 0):
            raise ValueError(
                f"Ma trận không hợp lệ: {matrix_path.name}"
            )

        support = matrix.sum(axis=1, keepdims=True)

        normalized = np.divide(
            matrix.astype(float),
            support,
            out=np.zeros((5, 5), dtype=float),
            where=support > 0,
        )

        figure, axes = plt.subplots(
            1, 2,
            figsize=(13, 5.5),
            constrained_layout=True,
        )

        for axis, values, title, is_normalized in [
            (axes[0], matrix, "Số nhịp", False),
            (axes[1], normalized, "Tỷ lệ theo nhãn thật", True),
        ]:
            image = axis.imshow(
                values,
                cmap="Blues",
                vmin=0,
                vmax=1 if is_normalized else None,
            )

            threshold = (
                0.5 if is_normalized
                else float(matrix.max()) / 2
            )

            for row in range(5):
                for column in range(5):
                    label = (
                        f"{values[row, column]:.1%}"
                        if is_normalized
                        else f"{int(values[row, column]):,}"
                    )

                    axis.text(
                        column,
                        row,
                        label,
                        ha="center",
                        va="center",
                        fontsize=10,
                        color=(
                            "white"
                            if values[row, column] > threshold
                            else "black"
                        ),
                    )

            axis.set_xticks(range(5), class_names)
            axis.set_yticks(range(5), class_names)
            axis.set_xlabel("Nhãn dự đoán")
            axis.set_ylabel("Nhãn thật")
            axis.set_title(title)

            figure.colorbar(
                image,
                ax=axis,
                label="Tỷ lệ" if is_normalized else "Số nhịp",
                shrink=0.85,
            )

        figure.suptitle(
            f"{model_name} — Ma trận nhầm lẫn DS2 — seed {seed}",
            fontsize=14,
            fontweight="bold",
        )

        filename = f"confusion_ds2_{variant}_seed{seed}"

        figure.savefig(
            FIG_DIR / f"{filename}.png",
            dpi=300,
            bbox_inches="tight",
        )

        figure.savefig(
            FIG_DIR / f"{filename}.pdf",
            bbox_inches="tight",
        )

        if seed == 42:
            plt.show()

        plt.close(figure)
        saved_count += 1

        print(
            f"{model_name} | seed {seed} | "
            f"{int(matrix.sum()):,} nhịp | Đã lưu hình"
        )

print(
    f"\nĐã lưu {saved_count} hình PNG "
    f"và {saved_count} bản PDF."
)
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

metrics = pd.read_csv(
    OUT_DIR / "results" / "ds2_metrics.csv"
)

model_order = ["baseline", "cnn_rr"]
model_names = {
    "baseline": "CNN cơ sở",
    "cnn_rr": "CNN + RR",
}
colors = {
    "baseline": "#2563EB",
    "cnn_rr": "#EA580C",
}

metric_columns = [
    "f1_macro", "sp_macro", "accuracy",
    "se_S", "ppv_S", "se_V", "ppv_V",
]

required_columns = {
    "model_variant", "seed", *metric_columns
}

if not required_columns.issubset(metrics.columns):
    raise ValueError(
        f"Thiếu cột: {sorted(required_columns - set(metrics.columns))}"
    )

metrics = metrics[
    metrics["model_variant"].isin(model_order)
].copy()

for variant in model_order:
    rows = metrics[metrics["model_variant"] == variant]

    if (
        len(rows) != 3
        or sorted(rows["seed"].astype(int).tolist()) != [42, 123, 2026]
    ):
        raise ValueError(
            f"{variant} phải có đúng ba seed 42, 123, 2026."
        )

    if not np.isfinite(
        rows[metric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"{variant} có chỉ số không xác định."
        )

means = (
    metrics.groupby("model_variant")[metric_columns]
    .mean()
    .reindex(model_order)
)

stds = (
    metrics.groupby("model_variant")[metric_columns]
    .std(ddof=1)
    .reindex(model_order)
)

plot_groups = [
    (
        ["f1_macro", "sp_macro", "accuracy"],
        ["F1 macro", "Sp macro", "Accuracy"],
        "So sánh chỉ số tổng hợp trên DS2",
        "ds2_overall_comparison",
    ),
    (
        ["se_S", "ppv_S", "se_V", "ppv_V"],
        ["Se S", "PPV S", "Se V", "PPV V"],
        "So sánh chỉ số lớp S và V trên DS2",
        "ds2_sv_comparison",
    ),
]

for columns, labels, title, filename in plot_groups:
    positions = np.arange(len(columns))
    width = 0.36

    figure, axis = plt.subplots(
        figsize=(11, 6),
        constrained_layout=True,
    )

    for model_index, variant in enumerate(model_order):
        values = means.loc[variant, columns].to_numpy(dtype=float)
        errors = stds.loc[variant, columns].to_numpy(dtype=float)
        bar_positions = positions + (model_index - 0.5) * width

        axis.bar(
            bar_positions,
            values,
            width=width,
            yerr=errors,
            capsize=5,
            color=colors[variant],
            label=model_names[variant],
            error_kw={"elinewidth": 1.3},
        )

        for position, value, error in zip(
            bar_positions, values, errors
        ):
            axis.text(
                position,
                value + error + 0.018,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=10,
            )

    axis.set_xticks(positions, labels)
    axis.set_ylim(0, 1.10)
    axis.set_yticks(np.arange(0, 1.01, 0.1))
    axis.set_ylabel("Giá trị chỉ số")
    axis.set_title(
        title + "\nTrung bình ± SD qua ba seed: 42, 123, 2026",
        fontsize=13,
        pad=16,
    )
    axis.legend(
        loc="upper left",
        bbox_to_anchor=(0, 1.01),
        ncol=2,
        frameon=False,
    )
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    figure.savefig(
        FIG_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIG_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

print("F1 macro trên DS2:")

for variant in model_order:
    print(
        f"{model_names[variant]}: "
        f"{means.loc[variant, 'f1_macro']:.4f} ± "
        f"{stds.loc[variant, 'f1_macro']:.4f}"
    )

print("\nĐã lưu thêm 2 hình PNG và 2 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ablation = pd.read_csv(
    OUT_DIR / "results" / "ds2_ablation.csv"
).sort_values("seed")

required = {
    "seed", "baseline_f1", "cnn_rr_f1", "delta_f1"
}

if not required.issubset(ablation.columns):
    raise ValueError(
        f"Thiếu cột: {sorted(required - set(ablation.columns))}"
    )

if (
    len(ablation) != 3
    or ablation["seed"].astype(int).tolist() != [42, 123, 2026]
):
    raise ValueError("Cần đúng ba seed 42, 123, 2026.")

values = ablation[
    ["baseline_f1", "cnn_rr_f1", "delta_f1"]
].to_numpy(dtype=float)

if not np.isfinite(values).all():
    raise ValueError("Bảng ablation có giá trị không hữu hạn.")

np.testing.assert_allclose(
    ablation["delta_f1"],
    ablation["cnn_rr_f1"] - ablation["baseline_f1"],
    rtol=1e-6,
    atol=1e-8,
)

delta_mean = ablation["delta_f1"].mean()
delta_std = ablation["delta_f1"].std(ddof=1)

figure, axes = plt.subplots(
    1, 2,
    figsize=(12, 5.5),
    constrained_layout=True,
)

seed_colors = ["#2563EB", "#EA580C", "#15803D"]

for color, row in zip(seed_colors, ablation.itertuples(index=False)):
    axes[0].plot(
        [0, 1],
        [row.baseline_f1, row.cnn_rr_f1],
        marker="o",
        markersize=7,
        linewidth=2,
        color=color,
        label=f"Seed {int(row.seed)}",
    )

axes[0].set_xticks([0, 1], ["CNN cơ sở", "CNN + RR"])
axes[0].set_xlim(-0.2, 1.2)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("F1 macro trên DS2")
axes[0].set_title("So sánh theo cùng seed")
axes[0].legend()

positions = np.arange(len(ablation))

bars = axes[1].bar(
    positions,
    ablation["delta_f1"],
    color=seed_colors,
    width=0.6,
)

for bar, value in zip(bars, ablation["delta_f1"]):
    axes[1].annotate(
        f"{value:+.4f}",
        xy=(bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 6 if value >= 0 else -6),
        textcoords="offset points",
        ha="center",
        va="bottom" if value >= 0 else "top",
    )

axes[1].axhline(0, color="black", linewidth=0.9)

axes[1].axhline(
    delta_mean,
    color="#7C3AED",
    linestyle="--",
    linewidth=1.5,
    label=f"Trung bình ΔF1 = {delta_mean:.4f}",
)

axes[1].set_xticks(
    positions,
    ablation["seed"].astype(int).astype(str),
)
axes[1].set_xlabel("Seed")
axes[1].set_ylabel("ΔF1 macro = CNN + RR − CNN cơ sở")
axes[1].set_title(
    f"Cải thiện khi thêm RR\n"
    f"Trung bình ± SD: {delta_mean:.4f} ± {delta_std:.4f}"
)

lower = min(0.0, float(ablation["delta_f1"].min()))
upper = max(0.0, float(ablation["delta_f1"].max()))
margin = max((upper - lower) * 0.25, 0.03)
axes[1].set_ylim(lower - margin, upper + margin)
axes[1].legend(loc="upper left", fontsize=9)

for axis in axes:
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

figure.suptitle(
    "Ablation nhánh RR trên DS2",
    fontsize=14,
    fontweight="bold",
)

figure.savefig(
    FIG_DIR / "ds2_rr_ablation.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIG_DIR / "ds2_rr_ablation.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print(ablation.to_string(index=False))
print(
    f"\nChênh lệch F1 macro trung bình ± SD: "
    f"{delta_mean:.4f} ± {delta_std:.4f}"
)
print("Đã lưu thêm 1 hình PNG và 1 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

conversion = pd.read_csv(
    OUT_DIR / "results" / "conversion_metrics.csv"
)

sizes = pd.read_csv(
    OUT_DIR / "results" / "model_sizes.csv"
)

with (OUT_DIR / "inference_config.json").open(
    encoding="utf-8"
) as file:
    inference_config = json.load(file)

export_seed = int(inference_config["seed"])

versions = [
    "PyTorch_FP32",
    "ONNX_FP32",
    "ONNX_INT8",
]

required_metrics = {
    "version", "f1_macro", "accuracy",
    "se_S", "ppv_S", "se_V", "ppv_V",
    "changed_labels", "label_agreement",
}

if not required_metrics.issubset(conversion.columns):
    raise ValueError(
        f"Thiếu cột: "
        f"{sorted(required_metrics - set(conversion.columns))}"
    )

if conversion["version"].duplicated().any():
    raise ValueError("Bảng chuyển đổi có phiên bản trùng.")

if sizes["version"].duplicated().any():
    raise ValueError("Bảng kích thước có phiên bản trùng.")

conversion = conversion.set_index("version").loc[versions]

size_versions = ["ONNX_FP32", "ONNX_INT8"]
sizes = sizes.set_index("version").loc[size_versions].copy()
sizes["KiB"] = sizes["bytes"] / 1024

metric_columns = [
    "f1_macro", "accuracy",
    "se_S", "ppv_S", "se_V", "ppv_V",
]

if not np.isfinite(
    conversion[metric_columns].to_numpy(dtype=float)
).all():
    raise ValueError("Bảng chuyển đổi có chỉ số không xác định.")

if not np.isfinite(sizes["KiB"]).all() or (sizes["KiB"] <= 0).any():
    raise ValueError("Kích thước mô hình không hợp lệ.")

version_labels = [
    "PyTorch\nFP32",
    "ONNX\nFP32",
    "ONNX\nINT8",
]

colors = ["#64748B", "#2563EB", "#EA580C"]

figure, axes = plt.subplots(
    1, 2,
    figsize=(12, 5.5),
    constrained_layout=True,
)

size_bars = axes[0].bar(
    ["ONNX FP32", "ONNX INT8"],
    sizes["KiB"].to_numpy(),
    color=[colors[1], colors[2]],
    width=0.55,
)

axes[0].bar_label(size_bars, fmt="%.2f", padding=5)
axes[0].set_ylabel("Kích thước tệp (KiB)")
axes[0].set_title("Kích thước mô hình")
axes[0].set_ylim(0, float(sizes["KiB"].max()) * 1.25)

f1_bars = axes[1].bar(
    version_labels,
    conversion["f1_macro"].to_numpy(),
    color=colors,
    width=0.55,
)

axes[1].bar_label(f1_bars, fmt="%.4f", padding=5)
axes[1].set_ylabel("F1 macro trên DS2")
axes[1].set_title("Chất lượng phân loại trước và sau chuyển đổi")
axes[1].set_ylim(0, 1)

for axis in axes:
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

figure.suptitle(
    f"CNN + RR — checkpoint seed {export_seed}",
    fontsize=14,
    fontweight="bold",
)

figure.savefig(
    FIG_DIR / "conversion_size_f1.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIG_DIR / "conversion_size_f1.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

columns = ["se_S", "ppv_S", "se_V", "ppv_V"]
labels = ["Se S", "PPV S", "Se V", "PPV V"]
positions = np.arange(len(columns))
width = 0.25

figure, axis = plt.subplots(
    figsize=(12, 6),
    constrained_layout=True,
)

for version_index, version in enumerate(versions):
    values = conversion.loc[
        version, columns
    ].to_numpy(dtype=float)

    bar_positions = positions + (version_index - 1) * width

    bars = axis.bar(
        bar_positions,
        values,
        width=width,
        color=colors[version_index],
        label=version.replace("_", " "),
    )

    axis.bar_label(
        bars,
        fmt="%.3f",
        padding=4,
        fontsize=9,
    )

axis.set_xticks(positions, labels)
axis.set_ylim(0, 1.12)
axis.set_yticks(np.arange(0, 1.01, 0.1))
axis.set_ylabel("Giá trị chỉ số trên DS2")
axis.set_title(
    f"Se và PPV lớp S, V trước và sau chuyển đổi\n"
    f"CNN + RR — checkpoint seed {export_seed}",
    fontsize=13,
    pad=16,
)
axis.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.01),
    ncol=3,
    frameon=False,
)
axis.grid(axis="y", alpha=0.25)
axis.set_axisbelow(True)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

figure.savefig(
    FIG_DIR / "conversion_sv_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIG_DIR / "conversion_sv_comparison.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

size_reduction = (
    1 - sizes.loc["ONNX_INT8", "bytes"]
    / sizes.loc["ONNX_FP32", "bytes"]
) * 100

f1_delta = (
    conversion.loc["ONNX_INT8", "f1_macro"]
    - conversion.loc["PyTorch_FP32", "f1_macro"]
)

print("So sánh chuyển đổi trên DS2:")
print(
    conversion[
        metric_columns + ["label_agreement", "changed_labels"]
    ].to_string()
)

print(f"\nINT8 giảm kích thước tệp: {size_reduction:.2f}%")
print(f"Chênh lệch F1 macro INT8 so với PyTorch: {f1_delta:+.6f}")
print("Đã lưu thêm 2 hình PNG và 2 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

with (OUT_DIR / "architecture.json").open(
    encoding="utf-8"
) as file:
    architecture = json.load(file)

parameter_counts = architecture["parameter_counts"]

plt.rcParams["font.family"] = "DejaVu Sans"

def draw_box(axis, x, y, width, height, text, color):
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.04",
        linewidth=1.3,
        edgecolor="#334155",
        facecolor=color,
    )
    axis.add_patch(box)
    axis.text(
        x + width / 2,
        y + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=11,
        linespacing=1.4,
    )

def draw_arrow(axis, start, end):
    axis.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 1.6,
            "color": "#334155",
            "shrinkA": 3,
            "shrinkB": 3,
        },
    )

def save_architecture(figure, filename):
    figure.savefig(
        FIG_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIG_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

encoder_text = (
    "Bộ mã hóa ECG\n"
    "Conv1D 1 → 16, kernel 7 + ReLU\n"
    "MaxPool1D 2\n"
    "Conv1D 16 → 32, kernel 5 + ReLU\n"
    "MaxPool1D 2\n"
    "Conv1D 32 → 64, kernel 3 + ReLU\n"
    "AdaptiveAvgPool1D 1 + Flatten\n"
    "Đặc trưng ECG: [B, 64]"
)

figure, axis = plt.subplots(figsize=(8, 10))
axis.set_xlim(0, 10)
axis.set_ylim(0, 10)
axis.axis("off")

draw_box(
    axis, 2.5, 8.6, 5, 0.9,
    "ECG đã chuẩn hóa\n[B, 1, 234]",
    "#DBEAFE",
)

draw_box(
    axis, 1.5, 5.2, 7, 2.9,
    encoder_text,
    "#EFF6FF",
)

draw_box(
    axis, 2.5, 2.9, 5, 1.6,
    "Bộ phân loại\n"
    "Linear 64 → 32 + ReLU\n"
    "Dropout 0.2\n"
    "Linear 32 → 5: logits [B, 5]",
    "#FFEDD5",
)

draw_box(
    axis, 2.5, 1.1, 5, 1.1,
    "Softmax khi suy luận\n"
    "Xác suất [B, 5]\n"
    "Thứ tự lớp: N, S, V, F, Q",
    "#DCFCE7",
)

draw_arrow(axis, (5, 8.6), (5, 8.1))
draw_arrow(axis, (5, 5.2), (5, 4.5))
draw_arrow(axis, (5, 2.9), (5, 2.2))

axis.set_title(
    f"CNN cơ sở\n"
    f"{parameter_counts['baseline']:,} tham số học được",
    fontsize=15,
    fontweight="bold",
    pad=15,
)

axis.text(
    5, 0.35,
    "B: số nhịp trong batch. Loss huấn luyện nhận logits.",
    ha="center",
    fontsize=10,
)

figure.tight_layout()
save_architecture(figure, "architecture_baseline")

figure, axis = plt.subplots(figsize=(9, 11))
axis.set_xlim(0, 10)
axis.set_ylim(0, 11)
axis.axis("off")

draw_box(
    axis, 0.25, 9.6, 4.3, 0.9,
    "ECG đã chuẩn hóa\n[B, 1, 234]",
    "#DBEAFE",
)

draw_box(
    axis, 5.45, 9.6, 4.3, 0.9,
    "RR đã chuẩn hóa\n[B, 3]",
    "#DBEAFE",
)

draw_box(
    axis, 0.25, 5.9, 4.3, 3.2,
    encoder_text,
    "#EFF6FF",
)

draw_box(
    axis, 5.45, 6.6, 4.3, 1.8,
    "Nhánh RR\n"
    "Linear 3 → 16 + ReLU\n"
    "Linear 16 → 16 + ReLU\n"
    "Đặc trưng RR: [B, 16]",
    "#EFF6FF",
)

draw_box(
    axis, 2.5, 4.4, 5, 0.9,
    "Ghép đặc trưng ECG và RR\n"
    "64 + 16 = 80 chiều: [B, 80]",
    "#F3E8FF",
)

draw_box(
    axis, 2.5, 2.3, 5, 1.5,
    "Bộ phân loại\n"
    "Linear 80 → 32 + ReLU\n"
    "Dropout 0.2\n"
    "Linear 32 → 5: logits [B, 5]",
    "#FFEDD5",
)

draw_box(
    axis, 2.5, 0.7, 5, 1,
    "Softmax khi suy luận\n"
    "Xác suất [B, 5]\n"
    "Thứ tự lớp: N, S, V, F, Q",
    "#DCFCE7",
)

draw_arrow(axis, (2.4, 9.6), (2.4, 9.1))
draw_arrow(axis, (7.6, 9.6), (7.6, 8.4))
draw_arrow(axis, (2.4, 5.9), (3.6, 5.3))
draw_arrow(axis, (7.6, 6.6), (6.4, 5.3))
draw_arrow(axis, (5, 4.4), (5, 3.8))
draw_arrow(axis, (5, 2.3), (5, 1.7))

axis.set_title(
    f"CNN + RR\n"
    f"{parameter_counts['cnn_rr']:,} tham số học được",
    fontsize=15,
    fontweight="bold",
    pad=15,
)

axis.text(
    5, 0.15,
    "B: số nhịp trong batch. Loss huấn luyện nhận logits.",
    ha="center",
    fontsize=10,
)

figure.tight_layout()
save_architecture(figure, "architecture_cnn_rr")

print("CNN cơ sở:", parameter_counts["baseline"], "tham số")
print("CNN + RR:", parameter_counts["cnn_rr"], "tham số")
print("Đã lưu thêm 2 sơ đồ PNG và 2 bản PDF.")
print("Thư mục:", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import hashlib
import zipfile
from IPython.display import FileLink, display

OUT_DIR = Path("/kaggle/working/TV2_RESTORED")
FIG_DIR = OUT_DIR / "figures_report"
SALIENCY_DIR = OUT_DIR / "saliency"

png_files = sorted(FIG_DIR.glob("*.png"))
pdf_files = sorted(FIG_DIR.glob("*.pdf"))
saliency_files = sorted(SALIENCY_DIR.glob("*.png"))

expected_figures = [
    "training_baseline_seed42_fold1",
    "training_cnn_rr_seed42_fold1",
    *[
        f"confusion_ds2_{variant}_seed{seed}"
        for variant in ["baseline", "cnn_rr"]
        for seed in [42, 123, 2026]
    ],
    "ds2_overall_comparison",
    "ds2_sv_comparison",
    "ds2_rr_ablation",
    "conversion_size_f1",
    "conversion_sv_comparison",
    "architecture_baseline",
    "architecture_cnn_rr",
]

missing = [
    f"{name}.{extension}"
    for name in expected_figures
    for extension in ["png", "pdf"]
    if not (FIG_DIR / f"{name}.{extension}").is_file()
]

if missing:
    raise FileNotFoundError(
        f"Chưa có đủ hình để đóng gói: {missing}"
    )

if len(saliency_files) != 12:
    raise ValueError(
        f"Cần 12 ảnh saliency, hiện tìm thấy {len(saliency_files)}."
    )

table_names = [
    "saliency_cases.csv",
    "ds2_metrics.csv",
    "ds2_summary.csv",
    "ds2_ablation.csv",
    "conversion_metrics.csv",
    "conversion_deltas.csv",
    "model_sizes.csv",
]

table_files = [
    OUT_DIR / "results" / name
    for name in table_names
]

missing_tables = [
    path.name for path in table_files
    if not path.is_file()
]

if missing_tables:
    raise FileNotFoundError(
        f"Thiếu bảng kết quả: {missing_tables}"
    )

package_files = (
    png_files
    + pdf_files
    + saliency_files
    + table_files
)

zip_path = Path("/kaggle/working/TV2_REPORT_FIGURES.zip")
manifest = {}

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in package_files:
        archive_name = path.relative_to(OUT_DIR).as_posix()
        archive.write(path, arcname=archive_name)

        manifest[archive_name] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

    archive.writestr(
        "sha256_manifest.json",
        json.dumps(
            manifest,
            ensure_ascii=False,
            indent=2,
        ),
    )

with zipfile.ZipFile(zip_path, "r") as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(
            f"Tệp lỗi trong ZIP: {bad_file}"
        )

print("Đóng gói thành công.")
print("Hình báo cáo PNG:", len(png_files))
print("Hình báo cáo PDF:", len(pdf_files))
print("Ảnh saliency:", len(saliency_files))
print("Bảng kết quả:", len(table_files))
print("Tên ZIP:", zip_path.name)
print("Đường dẫn:", zip_path)
print(
    "Kích thước:",
    round(zip_path.stat().st_size / (1024 ** 2), 2),
    "MiB",
)

display(FileLink("TV2_REPORT_FIGURES.zip"))